# Agriculture Sector Share vs Underemployment

Aligned with the project proposal **"Forecasting Underemployment in Sri Lanka Using Machine Learning"**, this combination explores the **Agricultural Sector Output/Employment share**, which is hypothesized to be the 4th-ranked predictor of underemployment spikes.

We are mapping the **Employment in agriculture (% of total employment)** dataset against the **Time-related underemployment proxy (Part-time employment %)**. Agricultural sector volatility generally influences casual/seasonal off-season employment stability.

## 1. Data Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr, pearsonr

# 1. Load Underemployment Proxy (Part-Time Employment)
part_time_df = pd.read_csv(
    "../labour/finalized_csv/Part_time_employment,_total_(% of total employment)_sl_indicators/Part time employment, total (% of total employment).csv"
)
part_time_df.set_index("Year", inplace=True)

# 2. Load Agriculture Sector Employment
agri_df = pd.read_csv(
    "../labour/finalized_csv/Employment_by_sector_(%)_sl_indicators/Employment in agriculture (% of total employment) (modeled ILO estimate).csv"
)
agri_df.set_index("Year", inplace=True)

# Align Data
underemp_df = pd.DataFrame({'underemp': part_time_df["Value"]}).dropna()
agriculture_df = pd.DataFrame({'agri_share': agri_df["Value"]}).dropna()

merged_data = underemp_df.join(agriculture_df, how='inner').reset_index()
merged_data.head()

## 2. Statistical Testing

In [ ]:
underemp_vals = merged_data['underemp'].values
agri_vals = merged_data['agri_share'].values

pearson_corr, p_p = pearsonr(agri_vals, underemp_vals)
spearman_corr, s_p = spearmanr(agri_vals, underemp_vals)

print("==================== Correlation Output ====================")
print(f"Data points:           {len(merged_data)} ({merged_data['Year'].min()}-{merged_data['Year'].max()})")
print(f"Pearson Correlation:   {pearson_corr:>7.4f} (p-value: {p_p:.5f})")
print(f"Spearman Correlation:  {spearman_corr:>7.4f} (p-value: {s_p:.5f})")
print("============================================================")

## 3. Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

color1 = 'forestgreen'
color2 = 'purple'

# Plot 1: Regression Scatter
ax1.scatter(agri_vals, underemp_vals, alpha=0.9, s=100, color=color1)
z = np.polyfit(agri_vals, underemp_vals, 1)
p = np.poly1d(z)
a_sorted = np.sort(agri_vals)
ax1.plot(a_sorted, p(a_sorted), "r--", linewidth=2, label='Linear fit')
ax1.set_xlabel('Agricultural Employment Share (%)', fontweight='bold')
ax1.set_ylabel('Underemployment Proxy (%)', fontweight='bold')
ax1.set_title(f'Agriculture Sector vs Underemployment', fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Time Series Comparison
ax2.plot(merged_data['Year'], underemp_vals, marker='o', color=color2, linewidth=2, label='Underemployment')
ax2.set_ylabel('Underemployment (%)', color=color2, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color2)

ax2_twin = ax2.twinx()
ax2_twin.plot(merged_data['Year'], agri_vals, marker='s', color=color1, linestyle='--', linewidth=2, label='Agriculture')
ax2_twin.set_ylabel('Agriculture Share (%)', color=color1, fontweight='bold')
ax2_twin.tick_params(axis='y', labelcolor=color1)
ax2.set_title('Dynamics Comparison Overlay', fontweight='bold')

plt.tight_layout()
plt.show()